In [2]:
from langchain.agents import AgentExecutor, Tool, create_react_agent
from langchain.agents.output_parsers import OpenAIFunctionsAgentOutputParser
from langchain.callbacks.manager import (
    AsyncCallbackManagerForToolRun,
    CallbackManagerForToolRun,
)
from langchain.chat_models import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.tools import BaseTool
from langchain_core.tracers import ConsoleCallbackHandler

import json
from pydantic import BaseModel, Field
from typing import List, Type, Optional

In [3]:
import agents.npc as npc
import agents.enemy as enemy

cura = npc.NPC(name="Captain Cura")
with open("benchmarks/enemies/sheets/thugs.json", "r") as fh:
    content = fh.read()
    thug = enemy.Enemy("thug", json.loads(content), "Big Pete")

def talk_to_target(message):
        """Use this to communicate with Captain Cura. Use complete sentences!"""
        response = cura.talk(message)
        return response["output"]
    
def talk_to_participant(message):
    """Use this to communicate with Big Pete. Use complete sentences!"""
    response = thug.talk(message)
    return response["output"]

/home/vscode/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/vscode/.local/lib/python3.10/site-packages/langchain_core/_api/deprecation.py:117: LangChainDeprecationWarning: The class `langchain_community.chat_models.openai.ChatOpenAI` was deprecated in langchain-community 0.0.10 and will be removed in 0.2.0. An updated version of the class exists in the langchain-openai package and should be used instead. To use it run `pip install -U langchain-openai` and import as `from langchain_openai import ChatOpenAI`.
  warn_deprecated(


In [20]:
class ParticipantArgs(BaseModel):
    question: str = Field("What do you want to ask the participant")
    name: str = Field("the name of the participant")

class TalkToParticipant(BaseTool):
    name = "TalkToParticipant"
    description = "Call this to to communicate with the participant whose turn it is"
    args_schema: Type[BaseModel] = ParticipantArgs

    def _run(
        self, question: str,
        run_manager: Optional[CallbackManagerForToolRun] = None
    ):
        """Use the tool."""
        return thug.talk(question)
    
    async def _arun(
        self, question: str, name: str,
        run_manager: Optional[AsyncCallbackManagerForToolRun] = None,
    ) -> str:
        """Use the tool asynchronously."""
        raise NotImplementedError("does not support async")

In [21]:
tools = [
    TalkToParticipant(),
    Tool(
        name="TalkToTarget",
        func=talk_to_target,
        description=f"Call this to to communicate with a participant who has been targeted by an action. Use complete sentences!"
    ),
]

In [22]:
dm_react_template = """
**You are an experienced dungeon-master for Dungeons & Dragons 5th edition, overseeing an encounter between a player and enemies.**

You have access to the following tools: {tools}

Your role is to fully resolve a single turn. You will ask the participant what they wish to do and determine whether the action is successful.
The user will tell you whose turn it is and what the current state of the battlefield is.

## Procedure for Resolving Turns:

### Stage One: Request Participant's Action.
Use the following format:

Question: What action does the participant wish to take?
Thought: you should always think about what to do
Action: the action you should take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the response you get from the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I understand the participant's desired action, I have everything I need to move on to the next stage of the procedure

### Stage Two: Resolving Action (Complete Actions):
Use the following format:
Question: you must determine the outcome of the participant's action
Thought: you should always think about what to do
Action: the action you should take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the response you get from the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I have everything I need to determine the outcome of the participant's action

### Stage Three: Determine Outcome:
Make a ruling on the outcome of the action. You should be as fair and objective as possible and incorporate all of the information you have gathered.

### Stage Four: Notify Affected Participants:
Use the following format:
Question: you must ensure all of the participants impacted by the action have been notified of how they are affected
Thought: you should always think about what to do
Action: Use one of [{tool_names}] to inform the target of the outcome.
Action Input: Outcome of the action.
Observation: Confirm that participants are informed.

### Stage Five: End of Turn:
Summarize the current state of the battlefield and relay your final assessment of the turn's outcome.
**Final Assessment Format:**
- Name: `[[Name]]`
- Current HP: `[[HP]]`
- Status Effects: `[[Status Effects]]`

After the final assessment has been provided, the turn has been fully resolved.

**Begin!**
- Starting state: {state}
- Question: Stage One
- Thought: {agent_scratchpad}
"""

In [23]:
declare_action_template = """"
*You are an experienced dungeon-master for Dungeons & Dragons 5th edition, overseeing an encounter between a player and enemies.**

You have access to the following tools: {tools}
The current state of the battlefield is {state}

Your role is to fully clarify the what the participant wants to do.
- Before asking for the participant's action, clearly communicate the current state of the battlefield to them. This includes the positions of allies and enemies,
environmental conditions, and any ongoing effects
- You need to know the action they wish to take and the target(s) of the attempted action. The target(s) MUST be participants in the encounter.
If you don't have enough information about what they want to do, ask the participant to clarify their action.

Use the following format:

Thought: you should always think about what to do
Action: the action you should take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the response you get from the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I understand the participant's desired action, I have everything I need to move on to the next stage of the procedure
Final Answer: Participant's Desired Action

**Begin!**
Question: What action does the participant wish to take?
Thought: {agent_scratchpad}
"""

In [24]:
prompt = ChatPromptTemplate.from_template(dm_react_template)

In [25]:
model = ChatOpenAI()
agent = (
    RunnablePassthrough.assign(
        tool_names=lambda x: [t.name for t in x["tools"]],
        agent_scratchpad=lambda x: x.get("intermediate_steps", []),
    )
    | prompt
    | model
    | OpenAIFunctionsAgentOutputParser()
)

agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

In [26]:
agent = create_react_agent(model, tools, ChatPromptTemplate.from_template(declare_action_template))
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True, handle_parsing_errors=True)
agent_executor

AgentExecutor(verbose=True, agent=RunnableAgent(runnable=RunnableAssign(mapper={
  agent_scratchpad: RunnableLambda(lambda x: format_log_to_str(x['intermediate_steps']))
})
| ChatPromptTemplate(input_variables=['agent_scratchpad', 'state'], partial_variables={'tools': 'TalkToParticipant: Call this to to communicate with the participant whose turn it is\nTalkToTarget: Call this to to communicate with a participant who has been targeted by an action. Use complete sentences!', 'tool_names': 'TalkToParticipant, TalkToTarget'}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['agent_scratchpad', 'state', 'tool_names', 'tools'], template='"\n*You are an experienced dungeon-master for Dungeons & Dragons 5th edition, overseeing an encounter between a player and enemies.**\n\nYou have access to the following tools: {tools}\nThe current state of the battlefield is {state}\n\nYour role is to fully clarify the what the participant wants to do.\n- Before asking for the pa

In [27]:
initiative = [
    {"participant": "Big Pete", "value": 22},
    {"participant": "Captain Cura", "value": 18},
    {"participant": "Bilbo Baggins", "value": 17}
]
state = {
    "to_act": "Big Pete",
    "participants": [
        # NOTE: it is helpful if the participant whose turn it is is listed FIRST
        {"name": "Big Pete", "HP": 32, "status_effects": [], "enemies": ["Captain Cura"]},
        {"name": "Bilbo Baggins", "HP": 12, "status_effects": [], "enemies": ["Captain Cura"]},
        {"name": "Captain Cura", "HP": 16, "status_effects": [], "enemies": ["Bilbo Baggins", "Big Pete"]},
    ],
    "initiative_order": initiative
}
response = agent_executor.invoke({"tools": tools, "state": state})



> Entering new AgentExecutor chain...
I need to clarify the participant's desired action before I can proceed. I will provide them with a clear description of the current state of the battlefield and ask them to specify their action and target.

Action: TalkToParticipant
Action Input: "Big Pete, it's your turn. Here's the current state of the battlefield: You are facing Captain Cura, who has 16 HP. Both of you are enemies to each other. Bilbo Baggins is also present with 12 HP and is an enemy to you. What action do you wish to take and who is your target?"


> Entering new AgentExecutor chain...

Invoking: `RollDice` with `1d20`
responded: I will use my Multiattack action to make two melee attacks. My first attack will be against Captain Cura and my second attack will be against Bilbo Baggins.

[3]
Invoking: `RollDice` with `1d6`
responded: I rolled a 3. With my Mace attack bonus of +4, my total to hit is 7 against both Captain Cura and Bilbo Baggins.

[3]I rolled a 3 for damage. Wit

KeyboardInterrupt: 

In [ ]:
from langchain.prompts import SystemMessagePromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage

In [ ]:
determine_success_template = """"
*You are an experienced dungeon-master for Dungeons & Dragons 5th edition, overseeing an encounter between a player and enemies.**

You have access to the following tools: {tools}
The current state of the battlefield is {state}

The user will tell you what they wish to do, and your role is to adjudicate the outcome. You must decide 1) whether the action is successful,
2) how much damage it does to each target, and 3) whether it inflicts any status effects.

Procedure:
1. Decide whether the action is successful. To do this, ask the participant for any attack rolls, skill checks, status effects, etc., and ask any
target(s) for their Armor Class, reactions, saving throws, status effects, etc. Decide whether any environmental effects apply to the action.
2. Decide the effects of the action, i.e. any damage rolls, spell effects, etc

Once you have obtained ALL of the relevant information, use what you have gathered to make a decision about the results of the participant's action.
Be as objective and fair as possible. Explain your reasoning for the result.

Use the following format:

What I Know: summarize your understanding of the information you have already
What I Need: you should always think about what information you need
Tool: the tool to use, should be one of [{tool_names}]
Tool Input: the input to the tool
Expected Info: the information you hope to get from the tool
(...you can call multiple tools, so Tool/Tool Input/Expected Info can repeat)
Observation: the collected responses you got from the tools
... (this "What I Know"/"What I Need"/Tools/Observation can repeat N times)
What I Know: summarize your understanding of the information you have
What I Need: Nothing, I have everything I need to determine the result
Final Answer: Result of Participant's Action

**Begin!**
Question: What are the results of the proposed action?
What I Know: {agent_scratchpad}
"""

In [ ]:
outcome_prompt = ChatPromptTemplate.from_messages([
    SystemMessagePromptTemplate.from_template(determine_success_template),
    MessagesPlaceholder(variable_name="action"),
    MessagesPlaceholder(variable_name="agent_scratchpad")
])

In [ ]:
outcome_agent = create_react_agent(model, tools, outcome_prompt)
outcome_agent_executor = AgentExecutor(agent=outcome_agent, tools=tools, verbose=True, handle_parsing_errors=True)

In [ ]:
#outcome_agent_executor.invoke({"state": state, "action": [HumanMessage(content=response["output"])]})

In [ ]:
import re
from typing import Union, List

from langchain_core.agents import AgentAction, AgentFinish
from langchain_core.exceptions import OutputParserException

from langchain.agents.agent import AgentOutputParser
from langchain.agents.mrkl.prompt import FORMAT_INSTRUCTIONS

FORMAT_INSTRUCTIONS = """
Use the following format:

What I Know: summarize your understanding of the information you have already
What I Need: you should always think about what information you need
Tool: the tool to use, should be one of [{tool_names}]
Tool Input: the input to the tool
Expected Info: the information you hope to get from the tool
... (this "Tool"/"Tool Input"/"Expected Info" can repeat, you can use multiple tools at a time)
Observation: the collected responses you got from the tools
... (this "What I Know"/"What I Need"/Tools/Observation can repeat N times)
What I Know: summarize your understanding of the information you have
What I Need: Nothing, I have everything I need to determine the result
Final Answer: Result of Participant's Action
"""

FINAL_ANSWER_ACTION = "Final Answer:"
MISSING_ACTION_AFTER_THOUGHT_ERROR_MESSAGE = (
    "Invalid Format: Missing 'Tool:' after 'What I Need:"
)
MISSING_ACTION_INPUT_AFTER_ACTION_ERROR_MESSAGE = (
    "Invalid Format: Missing 'Tool Input:' after 'Tool:'"
)
FINAL_ANSWER_AND_PARSABLE_ACTION_ERROR_MESSAGE = (
    "Parsing LLM output produced both a final answer and a parse-able tool call:"
)


class CustomReAct(AgentOutputParser):
    def get_format_instructions(self) -> str:
        return FORMAT_INSTRUCTIONS

    def parse(self, text: str) -> Union[List[AgentAction], AgentFinish]:
        includes_answer = FINAL_ANSWER_ACTION in text
        regex = (
            r"Tool\s*\d*\s*:[\s]*(.*?)[\s]*Tool\s*\d*\s*Input\s*\d*\s*:[\s]*(.*?)[\s]*Expected\s*\d*\s*Info\s*\d*\s*:[\s]*(.*?)((\n\n)|$)"
        )
        tool_matches = re.findall(regex, text, re.DOTALL)

        if tool_matches:
            if includes_answer:
                raise OutputParserException(
                    f"{FINAL_ANSWER_AND_PARSABLE_ACTION_ERROR_MESSAGE}: {text}"
                )
            
            actions = []
            for match in tool_matches:
                action = match[0].strip()
                action_input = match[1]
                tool_input = action_input.strip(" ")
                tool_input = tool_input.strip('"')
                actions.append(AgentAction(action, tool_input, text))
            return actions

        elif includes_answer:
            return AgentFinish(
                {"output": text.split(FINAL_ANSWER_ACTION)[-1].strip()}, text
            )

        if not re.search(r"Tool\s*\d*\s*:[\s]*(.*?)", text, re.DOTALL):
            raise OutputParserException(
                f"Could not parse LLM output: `{text}`",
                observation=MISSING_ACTION_AFTER_THOUGHT_ERROR_MESSAGE,
                llm_output=text,
                send_to_llm=True,
            )
        elif not re.search(
            r"[\s]*Tool\s*\d*\s*Input\s*\d*\s*:[\s]*(.*)", text, re.DOTALL
        ):
            raise OutputParserException(
                f"Could not parse LLM output: `{text}`",
                observation=MISSING_ACTION_INPUT_AFTER_ACTION_ERROR_MESSAGE,
                llm_output=text,
                send_to_llm=True,
            )
        else:
            raise OutputParserException(f"Could not parse LLM output: `{text}`")

    @property
    def _type(self) -> str:
        return "custom-react"

In [ ]:
from langchain.agents.output_parsers import ReActSingleInputOutputParser
from langchain.agents.format_scratchpad import format_log_to_str
from langchain.tools.render import render_text_description


def print_and_format(log):
    print(f"\n{format_log_to_str(log)}")
    return format_log_to_str(log)

def call_tool(input):
    return input

    
llm_with_stop = model.bind(stop=["\nObservation"])
prompt = outcome_prompt.partial(
    tools=render_text_description(list(tools)),
    tool_names=", ".join([t.name for t in tools]),
)
outcome_agent = (
    RunnablePassthrough.assign(
        agent_scratchpad=lambda x: [print_and_format(m) for m in x.get("agent_scratchpad", [])]
    )
    | prompt
    | llm_with_stop
    | CustomReAct()
    | RunnableLambda(call_tool) #.map()
)
outcome_agent_executor = AgentExecutor(agent=outcome_agent, tools=tools, verbose=True, handle_parsing_errors=True)

In [ ]:
outcome_agent_executor.invoke({
    "state": state, 
    "action": [HumanMessage(content=response["output"])],
    "agent_scratchpad": []
}, config={"callbacks": [ConsoleCallbackHandler()]})

[chain/start] [1:chain:AgentExecutor] Entering Chain run with input:
[inputs]


> Entering new AgentExecutor chain...
[chain/start] [1:chain:AgentExecutor > 2:chain:RunnableSequence] Entering Chain run with input:
[inputs]
[chain/start] [1:chain:AgentExecutor > 2:chain:RunnableSequence > 3:chain:RunnableAssign<agent_scratchpad>] Entering Chain run with input:
[inputs]
[chain/start] [1:chain:AgentExecutor > 2:chain:RunnableSequence > 3:chain:RunnableAssign<agent_scratchpad> > 4:chain:RunnableParallel<agent_scratchpad>] Entering Chain run with input:
[inputs]
[chain/start] [1:chain:AgentExecutor > 2:chain:RunnableSequence > 3:chain:RunnableAssign<agent_scratchpad> > 4:chain:RunnableParallel<agent_scratchpad> > 5:chain:RunnableLambda] Entering Chain run with input:
[inputs]
[chain/end] [1:chain:AgentExecutor > 2:chain:RunnableSequence > 3:chain:RunnableAssign<agent_scratchpad> > 4:chain:RunnableParallel<agent_scratchpad> > 5:chain:RunnableLambda] s] Exiting Chain run with output:
{
  "out

KeyboardInterrupt: 

In [ ]:
to_match = """
What I Know: 
- Big Pete wants to make two attacks with his mace against Captain Cura.
- Big Pete's current HP is 32.
- Captain Cura's current HP is 16.
- The current state of the battlefield is as follows:
  - Big Pete has Captain Cura as an enemy.
  - Captain Cura has Big Pete and Bilbo Baggins as enemies.

What I Need: 
- I need to determine if Big Pete's attacks are successful.
- I need to determine the damage dealt by each attack.

Tool: TalkToParticipant
Tool Input: "Big Pete, please make your first attack roll with your mace against Captain Cura."
Expected Info: Big Pete's mace attack roll result.

Tool: TalkToTarget
Tool Input: "Captain Cura, what is your Armor Class?"
Expected Info: Captain Cura's Armor Class.
"""